# 04. Evaluator completion workbench

This notebook completes the reusable local evaluator after the first approved smoke run. The earlier notebooks established the system model, reviewed individual adapter components, and proved one complete contract-driven run. This notebook now closes the gaps between those reviewed pieces so the evaluator—not a notebook—can safely own its approved execution paths.

It is not a new experiment and does not rerun the smoke suite, create a cache, write a packet, change Kaggriculture source, or interpret a candidate ranking merely by being opened.


## Development autonomy envelope

The owner has approved this bounded evaluator-completion phase to proceed notebook-first without per-component approval. We may explain, prototype, promote, and simulator-free-test the remaining evaluator capabilities in the order below. We still stop for a new or revised experiment contract, a simulator rerun or baseline materialization outside an existing approval, a durable-artifact overwrite/reuse choice, a source-policy change, a scope expansion, or a failed guard that needs recovery.

That keeps the development flow continuous while preserving the decisions that actually change evaluation authority or external state.


In [1]:
import json
from pathlib import Path
import inspect
import sys


def locate_project_root(start: Path) -> Path:
    """Find the source-controlled project without trusting notebook cwd."""
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / "experiments/schema/experiment-spec-v1.schema.json").is_file():
            return directory
        nested_project = directory / "carrot-tuning-system"
        if (nested_project / "experiments/schema/experiment-spec-v1.schema.json").is_file():
            return nested_project
    raise RuntimeError("Could not locate the carrot-tuning-system project root.")


PROJECT_ROOT = locate_project_root(Path.cwd())
REPOSITORY_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(REPOSITORY_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from evaluator.runner import LocalCandidateBatch, LocalInnerLoopAdapter, LocalRunRequest

ADAPTER = LocalInnerLoopAdapter()


## Completion checklist

- [x] Contract, target, lifecycle, and bounded-control gates
- [x] Baseline planning, validation, and one-entry materialization helper
- [x] Declared candidate fast-path evaluation and mandatory clean audits
- [x] Exact-coverage packet builder and immutable writer
- [x] Characterize the current top-level runner boundary
- [x] Promote packet completion into a source-level orchestration boundary
- [x] Define fail-closed baseline preparation for baseline-only contracts
- [x] Define repeat and existing-artifact semantics without overwriting evidence
- [x] Add a clear command/API boundary and regression coverage for completed runs

Each unchecked item is one future notebook section, not permission to combine all remaining work into one change.


## 1. Characterize the current top-level runner boundary

Today, `LocalInnerLoopAdapter.run()` owns preflight, cache revalidation, and the exact candidate/audit batch. It returns a `LocalCandidateBatch`; the reviewed packet builder and immutable writer remain callable source helpers that the completed smoke-run notebook invoked separately.

This section establishes the gap precisely. It does not run the evaluator, create an artifact, or decide whether `run()` itself should write evidence. The next section will design that completion boundary before any source behavior changes.


In [2]:
def describe_current_runner_boundary(adapter: LocalInnerLoopAdapter) -> dict:
    """Return source-interface facts without constructing a request or running a match."""
    return {
        "run_signature": str(inspect.signature(adapter.run)),
        "run_return_type": adapter.run.__annotations__.get("return"),
        "candidate_batch_type": LocalCandidateBatch.__name__,
        "packet_builder_available": callable(adapter.build_local_smoke_evidence_packet),
        "packet_writer_available": callable(adapter.write_local_smoke_evidence_packet),
    }


CURRENT_RUNNER_BOUNDARY = describe_current_runner_boundary(ADAPTER)


In [3]:
def render_current_runner_boundary(boundary: dict) -> None:
    """Presentation only: render current source-interface facts."""
    from IPython.display import Markdown, display

    display(Markdown(
        f"""### Current evaluator boundary

| Concern | Current source behavior |
|:---|:---|
| `run()` return | `{boundary['candidate_batch_type']}` |
| Packet builder | **{boundary['packet_builder_available']}** — separate reviewed helper |
| Packet writer | **{boundary['packet_writer_available']}** — separate reviewed helper |
| Evaluator run triggered | No |
| Artifact written | No |

> The remaining design question is where completed-run orchestration belongs in source, while preserving explicit artifact and repeat safeguards.
"""
    ))


render_current_runner_boundary(CURRENT_RUNNER_BOUNDARY)


### Current evaluator boundary

| Concern | Current source behavior |
|:---|:---|
| `run()` return | `LocalCandidateBatch` |
| Packet builder | **True** — separate reviewed helper |
| Packet writer | **True** — separate reviewed helper |
| Evaluator run triggered | No |
| Artifact written | No |

> The remaining design question is where completed-run orchestration belongs in source, while preserving explicit artifact and repeat safeguards.


## 2. Complete one already-audited batch into immutable evidence

`complete_local_candidate_batch(batch, report_directory)` is the narrow source-level completion seam. Its input is a `LocalCandidateBatch` that `run()` has already preflighted and clean-audited. Its output is a frozen record containing that unchanged batch, the validated descriptive packet, and the immutable writer receipt.

The builder validates complete declared coverage before the writer is reached. Therefore an incomplete, duplicated, unaudited, or mismatched batch cannot create even a temporary evidence directory. If validation succeeds, the existing exclusive writer creates exactly one contract-named JSON packet; an existing destination is refused rather than reused or replaced.

This section uses a synthetic batch and a temporary directory only. It does not call `run()`, import the simulator, create a baseline cache, modify Kaggriculture, write the real smoke packet, decide a winner, or make repeat-run semantics automatic.


In [4]:
from tempfile import TemporaryDirectory
from types import SimpleNamespace

from evaluator.runner import complete_local_candidate_batch

# This compact fixture mirrors the already-measured smoke shapes, but it is
# deliberately synthetic: exercising a source boundary is not new game evidence.
COMPLETION_SPEC = json.loads((
    PROJECT_ROOT / 'experiments/approved/carrot_investment/'
    'seed_opportunity_candidate_evidence_smoke_v1.json'
).read_text(encoding='utf-8'))
COMPLETION_BINDING = SimpleNamespace(
    revision=COMPLETION_SPEC['target']['expected_revision'],
    shared_sha256=COMPLETION_SPEC['target']['expected_shared_sha256'],
    engine_sha256=COMPLETION_SPEC['target']['expected_engine_sha256'],
)
COMPLETION_BASELINES = {(42, 0): {'decision_reward': 8849.0}, (42, 1): {'decision_reward': 8809.0}}
COMPLETION_SHAPES = {
    10.0: ((0, 8849.0, 8729.0, None, 0), (1, 8809.0, 8789.0, None, 0)),
    40.0: ((0, 8849.0, 8729.0, None, 0), (1, 8809.0, 8789.0, None, 0)),
    60.0: ((0, 8849.0, 8729.0, None, 0), (1, 8809.0, 8789.0, None, 0)),
    80.0: ((0, 9776.0, 10846.0, 107, 612), (1, 9756.0, 10886.0, 107, 612)),
}

COMPLETION_RECORDS = []
for value, rows in COMPLETION_SHAPES.items():
    for seat, decision_reward, opponent_reward, divergence, suffix_actions in rows:
        outcome = {'decision_reward': decision_reward, 'opponent_reward': opponent_reward}
        COMPLETION_RECORDS.append({
            'candidate': {'name': 'seed_opportunity_value', 'old_value': 20.0, 'new_value': value},
            'match': {'seed': 42, 'decision_player': seat, 'steps': 720},
            'candidate_source_revision': COMPLETION_BINDING.revision,
            'mode': 'reused-baseline' if divergence is None else 'restored-suffix',
            'first_divergence_turn': divergence, 'suffix_actions': suffix_actions,
            'outcome': outcome, 'replay_elapsed_seconds': 0.0, 'suffix_elapsed_seconds': 0.0,
            'audit_status': 'passed',
            'clean_audit': {'outcome': outcome.copy(), 'actions_run': 719, 'elapsed_seconds': 0.0},
        })

SYNTHETIC_AUDITED_BATCH = LocalCandidateBatch(
    preflight=SimpleNamespace(specification=COMPLETION_SPEC, binding=COMPLETION_BINDING),
    baseline_outcomes=COMPLETION_BASELINES,
    audited_results=tuple(COMPLETION_RECORDS),
)

with TemporaryDirectory() as temporary_directory:
    COMPLETED_SYNTHETIC_RUN = complete_local_candidate_batch(
        SYNTHETIC_AUDITED_BATCH, Path(temporary_directory)
    )
    # A second call reaches the existing writer guard; it must not overwrite.
    try:
        complete_local_candidate_batch(SYNTHETIC_AUDITED_BATCH, Path(temporary_directory))
    except FileExistsError:
        COMPLETION_OVERWRITE_REFUSED = True
    else:
        raise AssertionError('completed-run seam unexpectedly permitted an overwrite')


In [5]:
def render_completed_batch_boundary(completed_run, overwrite_refused: bool) -> None:
    """Presentation only: render facts from the completed temporary fixture."""
    from IPython.display import Markdown, display

    display(Markdown(
        f"""### Completed-batch source boundary

| Fact | Result |
|:---|:---|
| Audited matches consumed | **{len(completed_run.batch.audited_results)}** |
| Packet matches | **{len(completed_run.evidence_packet.per_match)}** |
| All audits passed | **{completed_run.evidence_packet.all_audits_passed}** |
| Temporary packet bytes | `{completed_run.packet_write.byte_count}` |
| Duplicate write | **{'refused' if overwrite_refused else 'not checked'}** |

> This proves source behavior with synthetic data in a temporary directory. It is not a simulator run, durable evidence packet, or interpretation.
"""
    ))


render_completed_batch_boundary(COMPLETED_SYNTHETIC_RUN, COMPLETION_OVERWRITE_REFUSED)


### Completed-batch source boundary

| Fact | Result |
|:---|:---|
| Audited matches consumed | **8** |
| Packet matches | **8** |
| All audits passed | **True** |
| Temporary packet bytes | `5825` |
| Duplicate write | **refused** |

> This proves source behavior with synthetic data in a temporary directory. It is not a simulator run, durable evidence packet, or interpretation.


## 3. Why baseline-only preparation is not ready to orchestrate

The existing one-entry materializer is appropriately narrow: it accepts one planned `materialize` action and refuses a pre-existing path. What is still missing is a **machine-readable contract field** that says which operation class the contract authorizes. Both current approved contracts have the same schema shape and candidate list; their differing permission is expressed only in `purpose` prose.

An orchestrator must not infer permission by searching that prose, by recognizing an experiment identifier, or by assuming that a candidate-evidence contract may rebuild missing caches. It must fail closed until a future, versioned contract declares the allowed operation explicitly. Adding that field would require a schema and contract revision—a human-owned authorization decision—so this section records the guard rather than changing either completed contract.


In [6]:
BASELINE_PREPARATION_OPERATION = 'baseline_cache_preparation'


def require_declared_operation(specification: dict, operation: str) -> None:
    """Prototype the future machine-readable operation gate; no side effects."""
    allowed_operations = specification.get('authorized_operations')
    if not isinstance(allowed_operations, list):
        raise ValueError(
            'Contract does not declare machine-readable authorized_operations; '
            'refusing to infer execution authority from prose.'
        )
    if operation not in allowed_operations:
        raise ValueError(f'Contract does not authorize operation: {operation}')


BASELINE_ONLY_SPEC = json.loads((
    PROJECT_ROOT / 'experiments/approved/carrot_investment/'
    'seed_opportunity_baseline_smoke_v1.json'
).read_text(encoding='utf-8'))

try:
    require_declared_operation(BASELINE_ONLY_SPEC, BASELINE_PREPARATION_OPERATION)
except ValueError as error:
    BASELINE_PREPARATION_GUARD = str(error)
else:
    raise AssertionError('A v1 contract unexpectedly passed an undeclared operation gate')


In [7]:
def render_baseline_preparation_guard(message: str) -> None:
    """Presentation only: explain the intentionally fail-closed v1 result."""
    from IPython.display import Markdown, display

    display(Markdown(
        f"""### Baseline-only orchestration guard

| Question | Current answer |
|:---|:---|
| Machine-readable operation field in schema v1 | **No** |
| Safe inference from `purpose` prose | **No** |
| Baseline materialization reached | **No** |
| Guard result | `{message}` |

> This is the correct stop: a later versioned contract can grant one explicit operation, but the existing completed contracts are not silently broadened.
"""
    ))


render_baseline_preparation_guard(BASELINE_PREPARATION_GUARD)


### Baseline-only orchestration guard

| Question | Current answer |
|:---|:---|
| Machine-readable operation field in schema v1 | **No** |
| Safe inference from `purpose` prose | **No** |
| Baseline materialization reached | **No** |
| Guard result | `Contract does not declare machine-readable authorized_operations; refusing to infer execution authority from prose.` |

> This is the correct stop: a later versioned contract can grant one explicit operation, but the existing completed contracts are not silently broadened.


## 4. Resolve operation authority without revising completed evidence

The schema now permits an optional `execution_authorization.operations` list. The new `seed_opportunity_baseline_preparation_v2` record began as a draft successor, not an edit to the completed smoke contract. It declares only `baseline_cache_preparation`. The source-level gate reads this machine-readable list and rejects both missing authority and a different operation.

The promoted contract is now approved, but this component section remains a source-level authority demonstration: operation authority and lifecycle authority are separate gates. The actual approved cache inspection belongs in Notebook 05. No cache is planned, created, read, or replaced by this section.


In [8]:
from evaluator.runner import authorize_execution_operation

BASELINE_PREPARATION_DRAFT = json.loads((
    PROJECT_ROOT / 'experiments/approved/carrot_investment/'
    'seed_opportunity_baseline_preparation_v2.json'
).read_text(encoding='utf-8'))

DRAFT_BASELINE_AUTHORIZATION = authorize_execution_operation(
    BASELINE_PREPARATION_DRAFT, BASELINE_PREPARATION_OPERATION
)
DRAFT_CANDIDATE_AUTHORIZATION = authorize_execution_operation(
    BASELINE_PREPARATION_DRAFT, 'candidate_evidence_run'
)

if not DRAFT_BASELINE_AUTHORIZATION.is_authorized:
    raise AssertionError('draft should declare the baseline-preparation operation')
if DRAFT_CANDIDATE_AUTHORIZATION.is_authorized:
    raise AssertionError('baseline-only draft must not authorize candidate evaluation')
if BASELINE_PREPARATION_DRAFT['lifecycle_status'] != 'approved':
    raise AssertionError('the promoted contract must retain approved lifecycle status')


In [9]:
def render_operation_authority_result(baseline_authorization, candidate_authorization, lifecycle: str) -> None:
    """Presentation only: distinguish operation declaration from execution permission."""
    from IPython.display import Markdown, display

    display(Markdown(
        f"""### Draft operation authority

| Gate | Result |
|:---|:---|
| Baseline-cache preparation | **{baseline_authorization.is_authorized}** |
| Candidate-evidence run | **{candidate_authorization.is_authorized}** |
| Lifecycle | `{lifecycle}` — actual inspection is recorded separately in Notebook 05 |

> The contract shape now expresses scope precisely, but this is not an approved run and no simulator boundary was reached.
"""
    ))


render_operation_authority_result(
    DRAFT_BASELINE_AUTHORIZATION,
    DRAFT_CANDIDATE_AUTHORIZATION,
    BASELINE_PREPARATION_DRAFT['lifecycle_status'],
)


### Draft operation authority

| Gate | Result |
|:---|:---|
| Baseline-cache preparation | **True** |
| Candidate-evidence run | **False** |
| Lifecycle | `approved` — actual inspection is recorded separately in Notebook 05 |

> The contract shape now expresses scope precisely, but this is not an approved run and no simulator boundary was reached.


## 5. Plan one baseline-preparation action without running it

`LocalInnerLoopAdapter.plan_baseline_preparation(request)` is the source-level planner. It requires schema validity, target binding, approved lifecycle, explicit `baseline_cache_preparation` authority, local controls, and cache inspection before it can choose anything. It reads and validates every present cache; a present but invalid file is a hard diagnostic stop, not a candidate for replacement.

For an all-missing suite, it selects only the first contract-ordered seed/seat action. The returned selection is a frozen plan, not a simulator result. The separate one-entry materializer remains unwired, so this section cannot create a cache even when its synthetic plan says an action is missing.


In [10]:
from tempfile import TemporaryDirectory
from types import SimpleNamespace
from unittest.mock import patch

from evaluator.runner import BaselineCacheEntry, BaselineCacheState

# The approved contract supplies authority; this in-memory copy keeps cache
# facts synthetic so the component workbench cannot touch real caches.
SYNTHETIC_APPROVED_BASELINE_SPEC = dict(
    BASELINE_PREPARATION_DRAFT, lifecycle_status='approved'
)
SYNTHETIC_PREPARATION_BINDING = SimpleNamespace(
    revision=SYNTHETIC_APPROVED_BASELINE_SPEC['target']['expected_revision'],
    shared_sha256=SYNTHETIC_APPROVED_BASELINE_SPEC['target']['expected_shared_sha256'],
    engine_sha256=SYNTHETIC_APPROVED_BASELINE_SPEC['target']['expected_engine_sha256'],
)

with TemporaryDirectory() as temporary_directory:
    temporary_root = Path(temporary_directory)
    SYNTHETIC_ENTRIES = (
        BaselineCacheEntry(42, 0, temporary_root / 'seat-0.json.gz'),
        BaselineCacheEntry(42, 1, temporary_root / 'seat-1.json.gz'),
    )
    SYNTHETIC_CACHE_PLAN = SimpleNamespace(entries=SYNTHETIC_ENTRIES)
    SYNTHETIC_CACHE_STATE = BaselineCacheState(
        SYNTHETIC_CACHE_PLAN, (), SYNTHETIC_ENTRIES
    )
    SYNTHETIC_PREPARATION_REQUEST = LocalRunRequest(
        spec_path=temporary_root / 'draft.json',
        target_root=temporary_root / 'target',
        cache_root=temporary_root / 'baseline-caches',
        report_directory=temporary_root / 'reports',
    )

    # These patches replace target/cache facts only. The planner's lifecycle
    # and operation gates still run against the synthetic approved contract.
    with (
        patch.object(ADAPTER, 'read_and_validate_spec', return_value=SYNTHETIC_APPROVED_BASELINE_SPEC),
        patch.object(ADAPTER, 'bind_and_validate_target', return_value=SYNTHETIC_PREPARATION_BINDING),
        patch.object(ADAPTER, 'validate_local_controls', return_value=SimpleNamespace()),
        patch.object(ADAPTER, 'plan_baseline_caches', return_value=SYNTHETIC_CACHE_PLAN),
        patch.object(ADAPTER, 'inspect_baseline_cache_state', return_value=SYNTHETIC_CACHE_STATE),
    ):
        SYNTHETIC_PREPARATION_SELECTION = ADAPTER.plan_baseline_preparation(
            SYNTHETIC_PREPARATION_REQUEST
        )

if SYNTHETIC_PREPARATION_SELECTION.selected_action.entry != SYNTHETIC_ENTRIES[0]:
    raise AssertionError('planner must select the first missing contract-ordered entry')
if any(entry.path.exists() for entry in SYNTHETIC_ENTRIES):
    raise AssertionError('planning must not create a cache file')


In [11]:
def render_baseline_preparation_plan(selection) -> None:
    """Presentation only: render an already-computed synthetic selection."""
    from IPython.display import Markdown, display

    action = selection.selected_action
    display(Markdown(
        f"""### Baseline-preparation plan

| Fact | Result |
|:---|:---|
| Lifecycle gate | `{selection.lifecycle.declared_status}` |
| Operation gate | **{selection.operation.is_authorized}** |
| Reusable entries | `{len(selection.cache_preparation.reusable_actions)}` |
| Missing entries | `{len(selection.cache_preparation.materialize_actions)}` |
| Selected action | seed `{action.entry.seed}`, seat `{action.entry.seat}` |
| Cache created | **No — planning only** |

> This is a synthetic approved fixture. The real v2 inspection is recorded in Notebook 05; no materializer was called here.
"""
    ))


render_baseline_preparation_plan(SYNTHETIC_PREPARATION_SELECTION)


### Baseline-preparation plan

| Fact | Result |
|:---|:---|
| Lifecycle gate | `approved` |
| Operation gate | **True** |
| Reusable entries | `0` |
| Missing entries | `2` |
| Selected action | seed `42`, seat `0` |
| Cache created | **No — planning only** |

> This is a synthetic approved fixture. The real v2 inspection is recorded in Notebook 05; no materializer was called here.


## 6. Inspect repeat evidence destinations without reusing them

`inspect_local_smoke_evidence_packet_repeat(packet, report_directory)` is a read-only diagnostic companion to the exclusive writer. It compares the expected canonical packet bytes with the contract-named destination and reports one of three states: `missing`, `exact_match`, or `conflict`. It does not write, reuse, replace, or call the simulator.

An `exact_match` means only that identical bytes already exist. It is not a successful rerun, permission to skip an approved evaluation, or a substitute for a human decision about what to do next. A `conflict` is a hard stop: the existing file remains untouched. The writer still refuses either existing state.


In [12]:
from dataclasses import replace

from evaluator.runner import (
    inspect_local_smoke_evidence_packet_repeat,
    write_local_smoke_evidence_packet,
)

try:
    SYNTHETIC_REPEAT_PACKET = COMPLETED_SYNTHETIC_RUN.evidence_packet
except NameError as error:
    raise RuntimeError('Run section 2 before this temporary repeat-inspection fixture.') from error

with TemporaryDirectory() as temporary_directory:
    temporary_report_directory = Path(temporary_directory)
    REPEAT_MISSING = inspect_local_smoke_evidence_packet_repeat(
        SYNTHETIC_REPEAT_PACKET, temporary_report_directory
    )
    write_local_smoke_evidence_packet(SYNTHETIC_REPEAT_PACKET, temporary_report_directory)
    REPEAT_EXACT_MATCH = inspect_local_smoke_evidence_packet_repeat(
        SYNTHETIC_REPEAT_PACKET, temporary_report_directory
    )
    # A different expected packet exposes a conflict without modifying the existing file.
    DIFFERENT_SYNTHETIC_PACKET = replace(
        SYNTHETIC_REPEAT_PACKET,
        interpretation_status='synthetic alternate packet for repeat-inspection guard',
    )
    REPEAT_CONFLICT = inspect_local_smoke_evidence_packet_repeat(
        DIFFERENT_SYNTHETIC_PACKET, temporary_report_directory
    )
    try:
        write_local_smoke_evidence_packet(SYNTHETIC_REPEAT_PACKET, temporary_report_directory)
    except FileExistsError:
        REPEAT_WRITE_REFUSED = True
    else:
        raise AssertionError('exclusive writer unexpectedly permitted a duplicate')

if [REPEAT_MISSING.status, REPEAT_EXACT_MATCH.status, REPEAT_CONFLICT.status] != [
    'missing', 'exact_match', 'conflict'
]:
    raise AssertionError('repeat states did not remain distinct')


In [13]:
def render_repeat_inspection(missing, exact_match, conflict, write_refused: bool) -> None:
    """Presentation only: render read-only repeat facts from a temporary fixture."""
    from IPython.display import Markdown, display

    display(Markdown(
        f"""### Immutable packet repeat inspection

| Destination state | Result | Meaning |
|:---|:---|:---|
| No file | `{missing.status}` | A first exclusive write may be attempted |
| Same canonical bytes | `{exact_match.status}` | Visible for review; **not** a rerun success |
| Different expected bytes | `{conflict.status}` | Stop; do not reuse or overwrite |
| Duplicate writer call | **{'refused' if write_refused else 'unexpected'}** | Immutability remains enforced |

> These are temporary-fixture facts only. No real smoke packet was inspected, modified, reused, or overwritten.
"""
    ))


render_repeat_inspection(
    REPEAT_MISSING, REPEAT_EXACT_MATCH, REPEAT_CONFLICT, REPEAT_WRITE_REFUSED
)


### Immutable packet repeat inspection

| Destination state | Result | Meaning |
|:---|:---|:---|
| No file | `missing` | A first exclusive write may be attempted |
| Same canonical bytes | `exact_match` | Visible for review; **not** a rerun success |
| Different expected bytes | `conflict` | Stop; do not reuse or overwrite |
| Duplicate writer call | **refused** | Immutability remains enforced |

> These are temporary-fixture facts only. No real smoke packet was inspected, modified, reused, or overwritten.


## 7. Expose only named read-only evaluator commands

The reusable API remains the named adapter methods: `run()` for an already approved candidate batch, `plan_baseline_preparation()` for a no-write baseline plan, and `complete_local_candidate_batch()` for an already-audited batch. The command line deliberately exposes less: `preflight` and `baseline-plan` only.

Neither command imports the simulator, creates a cache, evaluates a candidate, writes a packet, or edits policy source. There is intentionally no broad `run`, `materialize`, or packet-write command that could accidentally turn a convenient shell invocation into an external side effect.


In [14]:
from contextlib import redirect_stdout
from io import StringIO

from evaluator.runner import build_cli_parser, main

CLI_PARSER = build_cli_parser()
CLI_COMMANDS = tuple(
    CLI_PARSER._subparsers._group_actions[0].choices  # Inspect only; no command runs here.
)
if set(CLI_COMMANDS) != {'preflight', 'baseline-plan'}:
    raise AssertionError(f'unexpected evaluator CLI commands: {CLI_COMMANDS}')

# Exercise the real preflight command against the existing approved contract.
# It reads JSON and lifecycle authority only; no target or simulator is requested.
preflight_output = StringIO()
with redirect_stdout(preflight_output):
    PREFLIGHT_EXIT_CODE = main([
        'preflight',
        str(PROJECT_ROOT / 'experiments/approved/carrot_investment/'
            'seed_opportunity_baseline_smoke_v1.json'),
    ])
PREFLIGHT_OUTPUT = preflight_output.getvalue()
if PREFLIGHT_EXIT_CODE != 0 or 'Stopped after lifecycle authorization' not in PREFLIGHT_OUTPUT:
    raise AssertionError('safe preflight command did not stop at the documented boundary')


In [15]:
def render_safe_cli_boundary(commands: tuple[str, ...], exit_code: int, output: str) -> None:
    """Presentation only: render facts from the completed preflight command."""
    from IPython.display import Markdown, display

    last_line = output.strip().splitlines()[-1]
    display(Markdown(
        f"""### Safe evaluator command boundary

| Concern | Result |
|:---|:---|
| Exposed commands | `{', '.join(commands)}` |
| Broad `run` command | **Absent** |
| Simulator/materializer CLI | **Absent** |
| Real approved-contract preflight | exit code `{exit_code}` |
| Preflight stop | {last_line} |

> This command surface is an inspection interface, not an execution shortcut.
"""
    ))


render_safe_cli_boundary(CLI_COMMANDS, PREFLIGHT_EXIT_CODE, PREFLIGHT_OUTPUT)


### Safe evaluator command boundary

| Concern | Result |
|:---|:---|
| Exposed commands | `preflight, baseline-plan` |
| Broad `run` command | **Absent** |
| Simulator/materializer CLI | **Absent** |
| Real approved-contract preflight | exit code `0` |
| Preflight stop | Stopped after lifecycle authorization; target and local-control validation were not requested. |

> This command surface is an inspection interface, not an execution shortcut.


## Evaluator completion record

This workbench has closed the reusable evaluator gaps without creating new game evidence. It now has explicit operation authorization, a one-action baseline planner, a completed-batch packet seam, immutable-repeat inspection, named safe CLI modes, and simulator-free regression coverage.

It does **not** authorize a simulator run, cache materialization, evidence-packet overwrite or reuse, policy-source change, candidate expansion, or a claim about which candidate is best. The owner subsequently promoted the baseline-preparation v2 contract and its real read-only plan found both anchored seed-42 caches reusable, so no action was selected or executed. Under future automation, an automated promotion or execution decision must be surfaced as `- [ ] Review automated decision: <description>` after it is made.
